<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
OutputParser
</b></font> </br></p>


---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Warum sind Parser wichtig?
---

LangChain stellt eine Vielzahl von Ausgabeparsern bereit, die speziell darauf ausgelegt sind, Informationen aus den Ergebnissen großer Sprachmodelle (LLMs) effizient zu extrahieren und zu strukturieren. Diese Parser sind essenzielle Bestandteile der LangChain-Architektur und häufig zentrale Elemente sogenannter LangChain-Ketten. Solche Ketten bestehen aus konfigurierbaren Abfolgen von Operationen, die Modellausgaben verarbeiten und für weiterführende Anwendungen nutzbar machen.


**Warum sind OutputParser so wichtig?**
+ LLMs geben standardmäßig unstrukturierte Texte zurück.
+ OutputParser sind nötig, um das LLM-Output strukturiert weiterzuverarbeiten.
+ Besonders bei komplexen Anwendungen (z. B. Ketten mit mehreren Modellen, Agenten oder RAG-Systemen) müssen die Antworten klar definiert sein.

**`with_structured_output()` als State-of-the-Art (LangChain 1.0+):**
+ Die Methode **`with_structured_output()`** ist die moderne Lösung für strukturierte Ausgaben seit LangChain 1.0+.
+ Nutzt die native **OpenAI Structured Output API**, die Schema-Konformität auf Modell-Ebene garantiert.
+ **Garantierte Zuverlässigkeit** durch automatisches Retry bei Schema-Verletzungen.
+ **Einfacher Code** ohne manuelle Format-Instruktionen im Prompt.
+ **Production-Ready** - Teil der stabilen LangChain 1.0+ API.

<p><font color='black' size="5">
Parser-Kategorien und ihr Status:
</font></p>

| Parser-Typ | Beispiele | Status | Verwendung |
|-----------|-----------|--------|------------|
| **Einfache Parser** | `StrOutputParser`, `JsonOutputParser` | ✅ **Essentiell** | Standard in LCEL-Chains |
| **Strukturierte Ausgabe** | `with_structured_output()` + Pydantic-Modell | ✅ **Kursstandard** | Typsichere Modellantworten ohne manuelles Parsen |
| **Spezial-Parser** | `OutputFixingParser`, `RetryParser` | 🔧 **Nische** | Fehlerbehandlung und Legacy-Migration |

Die Methode with_structured_output() wird durch mehrere Anbieter unterstützt ...



| Anbieter | Modelle mit Structured-Output-Support |
|----------|----------------------------------------|
| OpenAI | aktuelle GPT-5.x-Chatmodelle über `init_chat_model()` |
| Mistral | ausgewählte Mistral-Chatmodelle mit Tool-/JSON-Unterstützung |
| Google | Gemini-/VertexAI-Modelle mit strukturierter Ausgabe |
| Anthropic | Claude-Modelle mit Tool-/Funktionsschnittstellen |

# 2 | OutputParser (einfach)
---

<p><font color='black' size="5">
Basismodell
</font></p>

In [ ]:
# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

# ── Pydantic ──────────────────────────────────────────────────────────────────
from pydantic import BaseModel, Field

# ── Stdlib ────────────────────────────────────────────────────────────────────
from datetime import datetime

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.model_config import BASELINE

In [ ]:
# Modell initialisieren
llm = init_chat_model(BASELINE)

<p><font color='blue' size="4">
Ohne Nachbearbeitung mit einem Parser
</font></p>

In [ ]:
input = "Was ist Machine Learning?"
response = llm.invoke(input)

In [ ]:
for r in response:
    print(r)

<p><font color='blue' size="4">
Nachbearbeitung mit einem Parser
</font></p>

In [ ]:
parser = StrOutputParser()
mprint(parser.invoke(response))

# 3 | Datenformat Parser
---

LangChain bietet eine breite Palette an Parsern, die unterschiedliche Datenformate verarbeiten können, was seine Einsatzmöglichkeiten erheblich erweitert. Es unterstützt unter anderem die nahtlose Integration von Pandas-Datenrahmen, kommagetrennten Listen, JSON-Strukturen sowie Datums- und Zeitobjekten. Diese Flexibilität ermöglicht eine effiziente Anpassung an verschiedene Arten von Dateneingaben und macht LangChain zu einem leistungsstarken Werkzeug für die Analyse und Verarbeitung von Daten. Im Folgenden werden einige dieser Parser genauer betrachtet, ihre praktischen Anwendungen demonstriert und aufgezeigt, wie sie zur Optimierung von Prozessen und zur Gewinnung wertvoller Erkenntnisse beitragen können.

**Beispiel:** Die Ausgabe des LLM kann in einem JSON-Format strukturiert werden.

In [ ]:
#
# Variante: Prompt
#
prompt = (
    "Bitte gib mir eine fiktive Person mit Name und Alter im JSON-Format zurück, z.B. "
    '{"name": "Max", "age": 30}'
)

response = llm.invoke(prompt)

mprint("### 🤖 KI:")
mprint("---")
mprint(response.content)

In [ ]:
#
# Parser: JsonOutputParser
#

# Prompt
prompt = ChatPromptTemplate([
    ("system", "You are a creative Assistant."),
    ("user", "Give name and age of a person in JSON format."),
])

# Parser
parser = JsonOutputParser()

# Kette
chain = prompt | llm

# Ausführen
response = chain.invoke({})

mprint("### 🤖 KI - ohne Parser:")
mprint("---")
print(response)

mprint("### 🤖 KI - mit Parser:")
mprint("---")
parser.invoke(response)

# 4 | Structured Output (mittel)
---

Der **Structured Output** Ansatz in LangChain mit der Methode `with_structured_output()` ist die **state-of-the-art Lösung** für strukturierte LLM-Ausgaben in **LangChain**. Diese Methode nutzt die native Structured Output API von OpenAI, die garantiert, dass die Ausgabe dem definierten Schema entspricht.

**Vorteile von `with_structured_output()`:**
- **Garantierte Schema-Konformität**: OpenAI API erzwingt die Struktur auf Modell-Ebene
- **Kein weiterer Parser nötig**: Direkter Rückgabewert als Pydantic-Objekt
- **Höhere Zuverlässigkeit**: Automatisches Retry bei Schema-Verletzungen
- **Bessere Effizienz**: Keine zusätzlichen Format-Instruktionen im Prompt
- **Typ-Sicherheit**: Vollständige IDE-Unterstützung und Type-Hints
- **Production-Ready**: Teil der stabilen LangChain 1.0+ API
- **Best Practice**: Empfohlener Ansatz für alle strukturierten Outputs

Der erste Schritt besteht in der Definition eines **Pydantic BaseModel**, das die Struktur der erwarteten Ausgabe definiert. Anschließend wird das LLM mit `with_structured_output()` konfiguriert.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Verarbeitungsprozess</font> </br></p>
diagram = """
%%{init: {'theme':'light'}}%%

flowchart TD
    A["LLM.with_structured_output(PydanticModel)"] --> B["Konfiguriertes LLM"]
    B --> C["Prompt senden"]
    C --> D["LLM generiert strukturierten Output"]
    D --> E["Parsing + Validierung"]
    E --> F["Pydantic-Objekt"]
    F --> G["Weiterverarbeitung"]

"""
mermaid(diagram, width=400)

Pydantic ist eine Python-Bibliothek, die automatisch Daten validiert und konvertiert, indem sie Typ-Hinweise nutzt und Datenstrukturen deklarativ mit Python-Klassen zu definieren.


<p><font color='black' size="5">
4.1 | Text & Quelle
</font></p>

---

In [ ]:
# Pydantic-Modell definieren
class Response(BaseModel):
    user_input: str = Field(description="Frage des Benutzers")
    response: str = Field(description="Antwort auf die Frage des Benutzers")
    source: str = Field(description="Verwendete Quelle (Website) für die Antwort")

In [ ]:
Response.model_json_schema()

`with_structured_output()` erzeugt einen Wrapper um das LLM, der die Ausgabe nach einem vorgegebenen Schema strukturiert. Ein Wrapper ist eine Art *Hülle* um etwas anderes. Er verpackt eine bestehende Funktion, ein Programm oder eine Technik und macht sie einfacher oder anders nutzbar.

In [ ]:
# System-Prompt
system_prompt = """
Beantworte die folgende Frage so präzise wie möglich.
Gib auch die Quelle deiner Information an.
"""

# Chat-Prompt-Template mit externen Variablen
prompt = ChatPromptTemplate([
    ("system", "{system_prompt}"),
    ("user", "{user_input}")
])

user_input = "Was ist Machine Learning?"

In [ ]:
# Chain erstellen - kein Parser mehr nötig!
chain = prompt | llm.with_structured_output(Response)

In [ ]:
# Funktion zur Aufbereitung der Druck-Ausgabe
def markdown_out(response):
    mprint(f"### 🧑 Mensch")
    mprint(f"{response.user_input}")
    mprint(f"### 🤖 KI:")
    mprint(f"{response.response}")
    mprint(f"### Quelle:\n {response.source}")

In [ ]:
# Aufruf Chain mit Dictionary (ChatPromptTemplate)
parameter = {}
parameter["system_prompt"] = system_prompt
parameter["user_input"] = user_input
response = chain.invoke(parameter)

markdown_out(response)

<p><font color='black' size="5">
4.2 | Datum/Uhrzeit
</font></p>

---

Durch die Verwendung von Pydantics `datetime` Typ wird automatisch eine Validierung durchgeführt und das Datum in ein standardisiertes Python `datetime`-Objekt umgewandelt.

Dies ist besonders hilfreich für Anwendungen wie Terminplanung, Datenanalyse oder andere Bereiche, in denen eine präzise Verarbeitung zeitlicher Informationen erforderlich ist. Der moderne Ansatz mit `with_structured_output()` ist zuverlässiger als der veraltete `DatetimeOutputParser` und nutzt die garantierte Schema-Konformität der OpenAI API.

In [ ]:
# Pydantic-Modell für Datum/Uhrzeit (LangChain 1.0+ Ansatz)
class DateResponse(BaseModel):
    date: datetime = Field(description="Das extrahierte Datum als datetime-Objekt")
    explanation: str = Field(description="Kurze Erklärung zum Datum")

# Prompt-Template
prompt = ChatPromptTemplate([
    ("system", "<Task>\nBeantworte die Frage und gib das Datum zurück.\n</Task>\n\n<Instructions>\nWenn du das Datum nicht kennst, verwende 1111-01-01.\n</Instructions>"),
    ("user", "{user_input}")
])

Wir erstellen die Chain mit `with_structured_output()` - kein Parser mehr nötig!

In [ ]:
# Chain ohne Parser!
chain = prompt | llm.with_structured_output(DateResponse)

Wir werden nach zwei Daten fragen, einem realen und einem fiktiven.

In [ ]:
parameter = {}
parameter["user_input"] = "Wann wurde die Programmiersprache Python eingeführt?"
response = chain.invoke(parameter)

# Ergebnis: 20.02.1991 (oder 1991-02-20 als datetime)
print(f"Datum: {response.date}")
print(f"Erklärung: {response.explanation}")

In [ ]:
parameter = {}
parameter["user_input"] = "An welchem Tag startet der Große Krieg im Videospiel Fallout?"
response = chain.invoke(parameter)

# Ergebnis: 23.10.2077 (oder 2077-10-23 als datetime)
print(f"Datum: {response.date}")
print(f"Erklärung: {response.explanation}")

# 5 | Structured Output (komplex)
---

Nun wird ein Beispiel mit einer größeren Anzahl an Werten getestet. Das folgende Programm nimmt Texte in beliebigen Sprachen entgegen und übersetzt sie automatisch ins Deutsche, Englische, Spanische und Chinesische. Dank `with_structured_output()` ist die Struktur garantiert - ohne zusätzliche Format-Instruktionen.

**Schritt 1:** Definiere ein **Pydantic-Modell** für die erwartete Ausgabe. Mit `with_structured_output()` wird die Struktur garantiert.

In [ ]:
# Pydantic-Modell definieren
class TranslationResponse(BaseModel):
    detected: str = Field(description="Die Sprache der Nutzer-Eingabe")
    german: str = Field(description="German translation")
    english: str = Field(description="English translation")
    spanish: str = Field(description="Spanish translation")
    chinese: str = Field(description="Chinese translation")

**Schritt 2:** Erstelle einen **einfachen Prompt** - keine Format-Instruktionen mehr nötig!

In [ ]:
# ChatPromptTemplate
system_prompt = """
Übersetze in die angegebenen Sprachen.
"""

prompt = ChatPromptTemplate([
    ("system", "{system_prompt}"),
    ("user", "{user_input}")
])

**Schritt 3:** Chain ohne Parser - das strukturierte LLM gibt direkt ein typsicheres Pydantic-Objekt zurück!

In [ ]:
# LLM konfigurieren
llm = init_chat_model(BASELINE)

In [ ]:
# Chain - kein Parser mehr nötig!
chain = prompt | llm.with_structured_output(TranslationResponse)

Zunächst wird ein deutscher Satz getestet, um zu beobachten, wie die Übersetzung in die drei Zielsprachen erfolgt.

In [ ]:
# Beispiel 1: Deutscher Satz
parameter = {}
parameter["system_prompt"] = system_prompt
parameter["user_input"] = "Wann wurde Python eingeführt?"

response = chain.invoke(parameter)

# Ausgabe über Pydantic-Attribute
print(f"{'detected':10s}:  {response.detected}")
print(f"{'german':10s}:  {response.german}")
print(f"{'english':10s}:  {response.english}")
print(f"{'spanish':10s}:  {response.spanish}")
print(f"{'chinese':10s}:  {response.chinese}")

Dann wird ein englischer Satz getestet, um zu beobachten, wie die Übersetzung in die drei Zielsprachen erfolgt.

In [ ]:
# Beispiel 2: Englischer Satz
parameter = {}
parameter["system_prompt"] = system_prompt
parameter["user_input"] = "Who rides so late through night and wind?"

response = chain.invoke(parameter)

# Ausgabe über Pydantic-Attribute
print(f"{'detected':10s}:  {response.detected}")
print(f"{'german':10s}:  {response.german}")
print(f"{'english':10s}:  {response.english}")
print(f"{'spanish':10s}:  {response.spanish}")
print(f"{'chinese':10s}:  {response.chinese}")

# 6 | Eigene Parser
---

In bestimmten Szenarien kann es sinnvoll sein, einen benutzerdefinierten Parser zu erstellen, um die Modellausgabe eindeutig zu formatieren. Dafür bietet sich die Verwendung von **RunnableLambda** oder **RunnableGenerator** in LCEL an, was für viele Fälle ein guten Ansatz ist.  

<p><font color='black' size="5">
Anwendungsfall
</font></p>



Large Language Models (LLMs) wie GPT-4 können Text generieren, der Code und erklärende Beschreibungen nahtlos vermischt. Dies kann zwar für Lern- und Dokumentationszwecke unglaublich nützlich sein, kann aber eine Herausforderung darstellen, wenn aus solchen Ausgaben mit gemischtem Inhalt nur der Code extrahiert und ausgeführt werden muss. Um dies zu beheben, implementieren wir eine einfache Funktion, die nicht-Python-Codezeilen aus einer gegebenen Textzeichenfolge entfernt.

Bei diesem Ansatz werden reguläre Ausdrücke verwendet, um Zeilen zu identifizieren und beizubehalten, die der typischen Python-Syntax entsprechen, während Zeilen verworfen werden, die beschreibender Text zu sein scheinen. Aufgrund der inhärenten Komplexität und Variabilität sowohl von Python-Code als auch von natürlicher Sprache kann diese Methode jedoch nie perfekt sein. Sie basiert auf heuristischen Mustern, die Code manchmal fälschlicherweise als Text klassifizieren oder umgekehrt.

Im nächsten Abschnitt werden wir untersuchen, wie ein anderes LLM beim Entfernen von Nicht-Python-Code helfen kann und möglicherweise eine ausgefeiltere und genauere Lösung bietet. Das folgende Beispiel enthält eine Mischung aus LLM-Kommentaren und generiertem Code.









In [ ]:
# Benutzer-Funktion zur Trennung von Erläuterung und Code
def parse_ai_message(ai_message):
    """Trennt die Erläuterung und den Code aus einer AIMessage und gibt sie separat zurück."""
    text = ai_message.content  # Extrahiere den reinen Textinhalt der AIMessage

    if "```" in text:
        # Trennen der Erläuterung und des Codes
        parts = text.split("```")
        explanation = parts[0].strip()
        code = parts[1].strip() if len(parts) > 1 else ""
    else:
        # Falls kein Codeblock vorhanden ist, geben wir nur die Erläuterung zurück
        explanation = text.strip()
        code = ""

    return explanation, code  # Rückgabe von zwei separaten Werten

In [ ]:
# RunnableLambda für das Parsing erstellen
parser = RunnableLambda(parse_ai_message)

In [ ]:
# Eingabe an die Pipeline senden und Ergebnis ausgeben
user_input = """
    Erkläre mir, wie man eine einfache Funktion in Python.
    Erstelle ein Python-Code-Beispiel zur Berechnung von Primzahlen.
    Verwende bei Formeln das Format $ Formel $
"""

In [ ]:
# LLM und Parser verketten
chain = llm | parser

In [ ]:
explanation, code = chain.invoke(user_input)

In [ ]:
mprint("### 🤖 Erläuterung")
mprint(explanation)
mprint("### 🤖 Code")
print(code)

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> Wichtig ist nur: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**

Erzeuge eine einfache strukturierte Ausgabe mit `with_structured_output()` — ein Dictionary mit 2–3 Feldern für einen Buchvorschlag (Titel, Autor, Genre).

**✅ Erledigt wenn:** `type(response)` zeigt das selbst definierte Pydantic-Modell — kein `str`, kein `dict`.

In [ ]:
# Grundlagen: Strukturierte Ausgabe mit with_structured_output()
# Startpunkt: Beispiel aus Kapitel 8.2 als Vorlage

# 1. Pydantic-Modell definieren (Titel, Autor, Genre)
# class Buchvorschlag(BaseModel):
#     titel: str
#     autor: str
#     genre: str

# 2. LLM mit structured output verbinden
# structured_llm = llm.with_structured_output(Buchvorschlag)

# 3. Aufrufen und type(response) prüfen
# ...

**Aufbau**

Parse eine JSON-Ausgabe für einen realen Fall (z.B. Produktbewertung mit Titel, Bewertung, Kategorie) sauber und extrahiere die Schlüsselwerte.

**✅ Erledigt wenn:** Das geparste Objekt enthält alle erwarteten Felder mit korrekten Python-Typen (`str`, `int`, `float`).

In [ ]:
# Aufbau: JSON-Parsing für Produktbewertung
# Startpunkt: Pydantic-Modell aus der Grundlagen-Aufgabe erweitern

# Felder: Titel, Bewertung (int 1-5), Kategorie, Kurzbeschreibung
# 1. Pydantic-Modell mit Field-Annotationen
# 2. structured_llm aufrufen
# 3. Felder und Typen ausgeben
# ...

**Vertiefung**

Vergleiche zwei Parser-Varianten (`JsonOutputParser` vs. `with_structured_output()`), provoziere einen Fehlerfall und ergänze Pydantic-Validierung für ein Feld.

**✅ Erledigt wenn:** Der provozierte Fehlerfall wirft eine klare Exception — kein stiller Fehler; Pydantic meldet Validierungsfehler mit verständlicher Meldung.

In [ ]:
# Vertiefung: Parser-Vergleich + Fehlerfall provozieren

# 1. JsonOutputParser vs. with_structured_output() — denselben Prompt beide Male
# 2. Fehlerfall: Pflichtfeld weglassen oder falschen Typ erzwingen
#    → try/except um ValidationError zeigen
# 3. Pydantic-Validator für ein Feld (z.B. Bewertung muss 1–5 sein)
# ...

# B | Dokumente zum Weiterlesen
---




📚 Ergänzende Artikel aus der Kurs-Dokumentation:

- [Strukturierte Ausgaben](https://ralf-42.github.io/GenAI/03-grundlagen/strukturierte-ausgaben.html)
- [LangChain 1.0+ Best Practices](https://ralf-42.github.io/GenAI/06-frameworks/langchain-best-practices.html)